### Frame differenciating method

In [7]:
# Libraries
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import time

In [8]:
# Functions

def import_frames(folder_path):
    image_list = []
    
    # Ensure the folder path is valid
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return image_list
    
    # Get a sorted list of filenames in the folder
    sorted_filenames = sorted(os.listdir(folder_path))
    
    # Iterate through sorted files in the folder
    for filename in sorted_filenames:
        file_path = os.path.join(folder_path, filename)
        
        # Check if the file is an image (you can add more image extensions if needed)
        if file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            # Read the image
     
            img = cv2.imread(file_path)
            
            # Append the image to the list
            if img is not None:
                gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                image_list.append(gray_img)
            else:
                print(f"Error reading image: {filename}")
    
    return image_list



def create_video(images_lists, text_list, output_path, fps=24):
    # Get the height and width of the frames
    height, width = images_lists[0][0].shape[:2]

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (2 * width, 2 * height), isColor=False)

    # Iterate through the frames in each list
    for i in range(len(images_lists[0])):
        # Create a 2 by 2 grid by concatenating images horizontally and vertically
        top_row = np.concatenate((images_lists[0][i], images_lists[1][i]), axis=1)
        bottom_row = np.concatenate((images_lists[2][i], images_lists[3][i]), axis=1)
        final_frame = np.concatenate((top_row, bottom_row), axis=0)
        
       
        # Add text to each space in the grid
        for j, text in enumerate(text_list):
            text_position = (width * (j % 2), height * (j // 2) + 20)
            cv2.putText(final_frame, str(text), text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
 
        # Write the frame to the video file
        video_writer.write(final_frame)

    # Release the VideoWriter object
    video_writer.release()



def dice_coef(groundtruth_mask, pred_mask):
    """Calculate Dice coefficient for similariy
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    total_sum = np.sum(pred_mask) + np.sum(groundtruth_mask)
    dice = np.mean(2*intersect/total_sum)
    if np.isnan(dice): # Ground truth and pred mask all zeros
        dice = 1.0
    return round(dice, 3) 

def iou_coef(groundtruth_mask, pred_mask):
    """Calculate IoU coefficient for similariy
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    union = np.sum(pred_mask) + np.sum(groundtruth_mask) - intersect
    iou = np.mean(intersect/union)
    if np.isnan(iou): # Ground truth and pred mask all zeros
        iou = 1.0
    return round(iou, 3)

def precision_score(groundtruth_mask, pred_mask):
    """Calculate precision score by pixel 
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    total_pixel_pred = np.sum(pred_mask)
    precision = np.mean(intersect/total_pixel_pred)
    if np.isnan(precision): # Ground truth and pred mask all zeros
        precision = 1.0
    return round(precision, 3)

def accuracy_score(groundtruth_mask, pred_mask):
    """Calculate accuracy score by pixel 
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    union = np.sum(pred_mask) + np.sum(groundtruth_mask) - intersect
    xor = np.sum(groundtruth_mask==pred_mask)
    acc = np.mean(xor/(union + xor - intersect))
    return round(acc, 3)


def remove_noise(binary_image, kernel_size=(5, 5)):
    kernel = np.ones(kernel_size, np.uint8)
    opening = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel)
    return opening

def create_background_model(frames):
    return np.median(frames, axis=0).astype(np.uint8)

def frame_differencing(frames, num_estimation_frames, group_size):
    """Frame differencing method"""
    # Getting the frames 
    background_estimation_frames = frames[:num_estimation_frames]
    # Initial background
    background_model = create_background_model(background_estimation_frames)
    diffed_frames = [] 

    for f in range(len(frames) - group_size):
        frame_group = []
        for i in range(group_size):
            # Difference between background and frame
            frame_group.append(cv2.absdiff(frames[i + f], background_model))
        # Sum all the resultant diffs
        diffed_frame = np.sum(frame_group, axis=0).astype(np.uint8)
        _, binary_image = cv2.threshold(diffed_frame, 30, 255, cv2.THRESH_BINARY)
        # Morfological noise reduction
        no_noise = remove_noise(binary_image)
        diffed_frames.append(no_noise)

    return diffed_frames


In [36]:
# Import input sequences
wdir = 'D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 2/pedestrians'
input_sequence_path = os.path.join(wdir, 'pedestrians', 'input') 
input_sequence = import_frames(input_sequence_path)

# First frames don't contain movement
input_sequence = input_sequence[400:]  

# Import ground truth sequences
groundtruth_sequence_path = os.path.join(wdir, 'pedestrians', 'groundtruth') 
groundtruth_sequence = import_frames(groundtruth_sequence_path)[400:]

# Getting the differenciating sequences
start_diff = time.time()
diffed_sequence = frame_differencing(input_sequence, 50, 8)
time_diff = round(time.time() - start_diff, 3)

# Create and save the video
output_video_path = os.path.join(wdir, 'output_video_diff.avi')
text_list = ["d", "c", "bb", "a"]


sequences = [input_sequence[:len(input_sequence)-8], groundtruth_sequence,
             input_sequence[:len(input_sequence)-8], diffed_sequence]

create_video(sequences, text_list, output_video_path, fps=24)
print(f"Video saved to: {output_video_path}")


# Convert ground truth sequence to binary
grt_sequence = []
for gt_frame in groundtruth_sequence:
    _, gt_bin = cv2.threshold(gt_frame, 30, 1, cv2.THRESH_BINARY)
    grt_sequence.append(gt_bin)

# Metrics
dice_tot = []
precision_tot = []
accuracy_tot = []
iou_tot = []

for gt_frame, dff_frame in zip(grt_sequence, diffed_sequence):
    dice = dice_coef(gt_frame, dff_frame)
    precision = precision_score(gt_frame, dff_frame)
    accuracy  = accuracy_score(gt_frame, dff_frame)
    iou = iou_coef(gt_frame, dff_frame)
    
    dice_tot.append(dice)
    precision_tot.append(precision)
    accuracy_tot.append(accuracy)
    iou_tot.append(iou)
    
print(f"Efficiency: {time_diff}")
print(f"Dice Coefficient: {round(np.mean(dice_tot),3)}")
print(f"Pixel Accuracy: {round(np.mean(accuracy_tot),3)}")
print(f"Pixel Precision: {round(np.mean(precision_tot),3)}")
print(f"IoU Metric: {round(np.mean(iou_tot),3)}")

Video saved to: D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 2/pedestrians\output_video_diff.avi
Efficiency: 1.237
Dice Coefficient: 0.588
Pixel Accuracy: 0.505
Pixel Precision: 0.295
IoU Metric: 0.503


In [32]:

sequences = [input_sequence[:len(input_sequence)-8], groundtruth_sequence[:len(input_sequence)-8], 
             list_black[:len(input_sequence)-8], diffed_sequence]

create_video(sequences, text_list, output_video_path, fps=24)
print(f"Video saved to: {output_video_path}")



Video saved to: D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 2/pedestrians\output_video_diff.avi


In [34]:
len(input_sequence[:len(input_sequence)-8]), len(groundtruth_sequence), len(list_black), len(diffed_sequence)

(691, 699, 699, 691)